# Data cleaning: movies, sales and normalization

**Author:** [Luca Alexnadrescu]

This notebook prepares the data for the MOVIES and SALES tables of our database,
and normalizes the directors, actors, genres and awards into separate tables. The
cleaning of the critic and user reviews is done by my team members in a separate
notebook.

## Input
- `data_raw/metaClean43Brightspace.xlsx` — film data from Metacritic (11,364 films)
- `data_raw/sales.xlsx` — box-office data from the-numbers.com (30,612 records)

## Output (in `data_clean/`)
- `movies.csv` — MOVIES table, 11,364 films
- `sales.csv` — SALES table, 8,829 films with box-office data
- `genres.csv`, `movie_genres.csv`, `directors.csv`, `movie_directors.csv`,
  `actors.csv`, `movie_actors.csv`, `awards.csv`, `movie_awards.csv` — the
  normalized attributes with their junction tables
- `movies_for_matching.csv` — helper file for linking sales to films (not a database table)

## Structure
1. Cells 1–2: environment check
2. Cells 3–6: cleaning the Metacritic data and creating the MOVIES table
3. Cells 7–15: linking the sales data to films on title and release year, and creating SALES
4. Cells 16–19: normalizing the multi-valued columns into lookup and junction tables
5. Cell 20: preparing the team's review file for import

## Requirements
pandas, ftfy, python-calamine. Run the cells in order from top to bottom; the
notebook writes no data to the database, only CSV files that are imported through
pgAdmin.

## Importing the libraries

This notebook uses three libraries. `os` lets Python look at folders and files
on the computer. `pandas` is used to read the data files and work with them as
tables (called DataFrames). `ftfy` repairs text in which special characters,
such as accented letters, were damaged when the data was saved.

In [1]:
import os            # to look inside folders on the computer
import pandas as pd  # to work with tables of data
import ftfy          # to repair damaged characters in text

## Checking the working environment

Before cleaning the data, we check that the notebook can find its files. The
raw data is stored in `data_raw` and the cleaned data is saved in `data_clean`.
Keeping the two separate means the original files are never changed, so the
cleaning can always be repeated from the start.

In [2]:
# Show which folder the notebook is running in
print("Working folder:", os.getcwd())

# Show the files in the raw and clean data folders
print("data_raw:", os.listdir("data_raw"))
print("data_clean:", os.listdir("data_clean"))

Working folder: c:\Users\lucax\OneDrive - HvA\Assignments\Database\Cleaning
data_raw: ['metaClean43Brightspace.xlsx', 'sales.xlsx']
data_clean: ['actors.csv', 'awards.csv', 'critics.csv', 'critic_reviews.csv', 'directors.csv', 'genres.csv', 'movies.csv', 'movies_for_matching.csv', 'movie_actors.csv', 'movie_awards.csv', 'movie_directors.csv', 'movie_genres.csv', 'sales.csv', 'users.csv', 'user_reviews.csv']


## Loading the Metacritic data

The Metacritic file contains one row per movie with its title, url and review
scores. We load it into a DataFrame called `movies_raw`, which we never change,
so it can always be compared with the cleaned version. The `calamine` engine is
used because it reads large Excel files much faster than the default one.

In [3]:
# Read the Excel file into a table
movies_raw = pd.read_excel("data_raw/metaClean43Brightspace.xlsx", engine="calamine")

# Show the number of rows and columns
print(movies_raw.shape)

# Show the first five rows
movies_raw.head()

(11364, 13)


,url,title,studio,rating,runtime,cast,director,genre,summary,awards,metascore,userscore,RelDate
0,https://www.metacritic.com/movie/!women-art-re...,!Women Art Revolution,Hotwire Productions,| Not Rated,83.0,NaN,Lynn Hershman-Leeson,Documentary,NaN,NaN,70,NaN,2011-06-01
1,https://www.metacritic.com/movie/10-cloverfiel...,10 Cloverfield Lane,Paramount Pictures,| PG-13,104.0,"John Gallagher Jr.,John Goodman,Mary Elizabeth...",Dan Trachtenberg,"Action,Sci-Fi,Drama,Mystery,Thriller,Horror","Waking up from a car accident, a young woman (...","#18MostDiscussedMovieof2016 , #1MostSharedMovi...",76,7.7,2016-03-11
2,https://www.metacritic.com/movie/10-items-or-less,10 Items or Less,Click Star,| R,82.0,"Jonah Hill,Morgan Freeman,Paz Vega",Brad Silberling,"Drama,Comedy,Romance",While researching a role as a supermarket mana...,NaN,54,5.8,2006-12-01
3,https://www.metacritic.com/movie/10-years,10 Years,Anchor Bay Entertainment,| R,100.0,"Channing Tatum,Chris Pratt,Jenna Dewan",Jamie Linden,"Drama,Comedy,Romance",NaN,NaN,61,6.9,2012-09-14
4,https://www.metacritic.com/movie/100-bloody-acres,100 Bloody Acres,Music Box Films,| Not Rated,91.0,NaN,Cameron Cairnes,"Horror,Comedy",Reg and Lindsay run an organic fertilizer busi...,NaN,63,7.5,2013-06-28


## Inspecting the data before cleaning

Before changing anything, we count the missing values and the duplicates. The
url has no duplicates, so every movie has a unique url; this is why the url is
the unique key (UK) of the MOVIES table in our ERD. The title cannot be used as
a key, because 219 titles appear more than once (different movies with the
same name).

In [4]:
# Count the missing values in each column
print(movies_raw.isna().sum())

# Count how many urls and titles appear more than once
print("Duplicate URLs:", movies_raw["url"].duplicated().sum())
print("Duplicate titles:", movies_raw["title"].duplicated().sum())

url             0
title           0
studio        350
rating       1067
runtime       255
cast         3702
director       14
genre          20
summary      5897
awards       6977
metascore       0
userscore    2105
RelDate         0
dtype: int64
Duplicate URLs: 0
Duplicate titles: 219


## Selecting and cleaning the MOVIES columns

We keep only the columns needed for our ERD, plus the release date and runtime,
which are needed later to match movies with the sales data. The url is cleaned
(spaces removed, lowercase, no slash at the end) so that it matches exactly
when we later link the review data to the movies. Titles with damaged
characters are repaired with `ftfy`, and two titles it cannot repair are fixed
by hand. Finally we check that no duplicates or broken titles are left and that
all scores are within their valid range.

In [5]:
# Keep only the columns we need
columns_to_keep = ["url", "title", "metascore", "userscore", "RelDate", "runtime"]
movies = movies_raw[columns_to_keep].copy()

# Clean the url: remove spaces, make lowercase, remove "/" at the end
movies["url"] = movies["url"].str.strip()
movies["url"] = movies["url"].str.lower()
movies["url"] = movies["url"].str.rstrip("/")

# Clean the title: remove spaces, then repair damaged characters
movies["title"] = movies["title"].str.strip()
movies["title"] = movies["title"].apply(ftfy.fix_text)

# Two titles that ftfy cannot repair, fixed by hand
movies["title"] = movies["title"].replace("Caterina va in cittÃ", "Caterina va in città")
movies["title"] = movies["title"].replace("Cet amour-lÃ", "Cet amour-là")

# Checks: all of these should be 0 or True
print("Duplicate URLs:", movies["url"].duplicated().sum())
print("Broken titles left:", movies["title"].str.contains("Ã").sum())
print("Metascore 0-100:", movies["metascore"].between(0, 100).all())
print("Userscore 0-10:", movies["userscore"].dropna().between(0, 10).all())

Duplicate URLs: 0
Broken titles left: 0
Metascore 0-100: True
Userscore 0-10: True


## Creating the primary key and saving the MOVIES table

Neither data file has a numeric id, so we create our own primary key,
`movie_id`, by numbering the movies from 1 upwards. We then save the MOVIES
table with exactly the columns of our ERD, in the same order. A second file,
`movies_for_matching.csv`, is saved as a helper for the next step; it is not a
database table. The files are saved as UTF-8 so that the repaired accents are
kept.

In [6]:
# Give every movie a number from 1 to 11364: this is the primary key
movies["movie_id"] = range(1, len(movies) + 1)

# The MOVIES table: only the columns from the ERD, in ERD order
movies_table = movies[["movie_id", "url", "title", "metascore", "userscore"]]
movies_table.to_csv("data_clean/movies.csv", index=False, encoding="utf-8")

# Helper file for matching with the sales data (not a database table)
matching_table = movies[["movie_id", "title", "RelDate", "runtime"]]
matching_table.to_csv("data_clean/movies_for_matching.csv", index=False, encoding="utf-8")

print(movies_table.shape)
print("data_clean:", os.listdir("data_clean"))

(11364, 5)
data_clean: ['actors.csv', 'awards.csv', 'critics.csv', 'critic_reviews.csv', 'directors.csv', 'genres.csv', 'movies.csv', 'movies_for_matching.csv', 'movie_actors.csv', 'movie_awards.csv', 'movie_directors.csv', 'movie_genres.csv', 'sales.csv', 'users.csv', 'user_reviews.csv']


## Loading the sales data

The sales file contains the box office figures, budget and runtime of each
movie. We also load the helper file from the MOVIES step, because every sales
row must be linked to a `movie_id`. The release date is read as a real date
(`parse_dates`), because we need the year for matching.

In [7]:
# Read the sales data
sales_raw = pd.read_excel("data_raw/sales.xlsx", engine="calamine")

# Read the helper file from the MOVIES step; RelDate is read as a date
movies_match = pd.read_csv("data_clean/movies_for_matching.csv", parse_dates=["RelDate"])

print(sales_raw.shape, movies_match.shape)

(30612, 16) (11364, 4)


## Removing unusable titles

We keep only the columns we need. Excel changed some movie titles, such as
"1921" or "02:22", into numbers or times, so the original title is lost and
these 78 rows cannot be matched to a movie; they are removed. The remaining
titles are repaired with `ftfy`, just like the Metacritic titles.

In [8]:
# Keep only the columns we need
columns_to_keep = ["year", "release_date", "title", "production_budget", "runtime",
                   "domestic_box_office", "international_box_office", "opening_weekend", "url"]
sales = sales_raw[columns_to_keep].copy()

# A function that checks if a value is text (a string)
def is_text(value):
    return isinstance(value, str)

# Check every title: True if it is text, False if Excel changed it
title_is_text = sales["title"].apply(is_text)
print("Titles Excel turned into numbers/dates:", (title_is_text == False).sum())

# Keep only the rows where the title is text
sales = sales[title_is_text].copy()

# Remove spaces and repair damaged characters
sales["title"] = sales["title"].str.strip()
sales["title"] = sales["title"].apply(ftfy.fix_text)

Titles Excel turned into numbers/dates: 78


## Creating a matching key

The sales data and the Metacritic data have no shared id, so movies have to be
matched on their title and year. The titles are written differently in the two
sources (for example "Walk to Remember, A" and "A Walk to Remember"), so the
function `make_key` turns every title into the same simple form: articles
moved to the front, no accents, lowercase and no punctuation. 14 Russian
titles become empty after this step and are removed, since they cannot be
matched.

In [9]:
import unicodedata   # needed to remove accents from letters

In [10]:

def make_key(title):
    key = title.strip()

    # Move the article to the front: "Walk to Remember, A" -> "A Walk to Remember"
    if key.endswith(", The"):
        key = "The " + key[:-5]
    elif key.endswith(", An"):
        key = "An " + key[:-4]
    elif key.endswith(", A"):
        key = "A " + key[:-3]

    # Remove accents: "Reykjavík" -> "Reykjavik"
    key = unicodedata.normalize("NFKD", key).encode("ascii", "ignore").decode()

    # Make lowercase and write "&" as "and"
    key = key.lower()
    key = key.replace("&", " and ")

    # Keep only letters and numbers; every other character becomes a space
    allowed = "abcdefghijklmnopqrstuvwxyz0123456789"
    clean_key = ""
    for character in key:
        if character in allowed:
            clean_key = clean_key + character
        else:
            clean_key = clean_key + " "

    # Remove double spaces: split into words, then join with one space
    words = clean_key.split()
    return " ".join(words)

# Make a key for every title in both tables
sales["title_key"] = sales["title"].apply(make_key)
movies_match["title_key"] = movies_match["title"].apply(make_key)

# Take the year from the Metacritic release date
movies_match["year"] = movies_match["RelDate"].dt.year

# Remove titles with an empty key
print("Titles with no usable key:", (sales["title_key"] == "").sum())
sales = sales[sales["title_key"] != ""]

# Show 10 random examples to check the keys
sales[["title", "title_key"]].sample(10)

Titles with no usable key: 14


,title,title_key
1444,Middleground,middleground
4283,Operator,operator
11092,Eliksir,eliksir
20825,Silent Hill,silent hill
20326,Leap Year,leap year
7269,Rodin,rodin
3240,Exhibition on Screen: Goya – Visions of Flesh ...,exhibition on screen goya visions of flesh and...
5264,Premiers crus,premiers crus
29038,Dev,dev
9393,Heli,heli


## Removing ambiguous movies

Some different movies share the same title and year (for example two movies
called "Adam" from 2020). In that case we cannot know which sales row belongs
to which movie, so all copies are removed instead of guessing. A wrong match
would give a movie the box office of another movie, which would make the
analysis unreliable.

In [11]:
# Mark every row whose title_key + year appears more than once
# keep=False marks ALL copies, not only the second one
sales_dup = sales.duplicated(["title_key", "year"], keep=False)
movies_dup = movies_match.duplicated(["title_key", "year"], keep=False)
print("Ambiguous sales rows:", sales_dup.sum(), "| Ambiguous movie rows:", movies_dup.sum())

# Keep only the rows that are NOT duplicates
sales_unique = sales[sales_dup == False]
movies_unique = movies_match[movies_dup == False]

Ambiguous sales rows: 59 | Ambiguous movie rows: 6


## Matching on title and year

We now join the sales data and the movies on the same title key and the same
year. An inner join keeps only the movies that exist in both sources. Each
matched sales row now receives the `movie_id` of its movie, which is the
foreign key in the SALES table.

In [12]:
# The columns from the movies we need for matching
movie_columns = movies_unique[["movie_id", "title_key", "year"]]

# Join: keep only rows with the same title_key AND the same year in both tables
exact = sales_unique.merge(movie_columns, on=["title_key", "year"], how="inner")

print("Exact matches:", len(exact))

Exact matches: 8367


## Matching movies with a one-year difference

Some movies have a different year in the two sources, for example because they
premiered at a festival in one year and were released in cinemas the next. For
the movies that were not matched yet, we therefore also accept a match when
the title is the same and the years differ by exactly one. Matches where one
sales row fits several movies (or the other way around) are removed, again to
avoid guessing.

In [13]:
# Find the rows that were NOT matched in the previous step
already_matched_sales = sales_unique["url"].isin(exact["url"])
already_matched_movies = movies_unique["movie_id"].isin(exact["movie_id"])
sales_left = sales_unique[already_matched_sales == False]
movies_left = movies_unique[already_matched_movies == False]

# Join on title only; the movie year gets the name "year_meta"
movie_columns = movies_left[["movie_id", "title_key", "year"]]
near = sales_left.merge(movie_columns, on="title_key", suffixes=("", "_meta"))

# Keep only matches where the years differ by exactly 1
near["year_difference"] = (near["year"] - near["year_meta"]).abs()
near = near[near["year_difference"] == 1]

# Remove matches where a sales row or a movie appears more than once
sales_twice = near["url"].duplicated(keep=False)
movie_twice = near["movie_id"].duplicated(keep=False)
ambiguous = sales_twice | movie_twice          # | means "or"
print("±1 year matches:", len(near), "| ambiguous, dropped:", ambiguous.sum())
near = near[ambiguous == False]

# Show 15 examples to check if the matches make sense
near[["title", "year", "year_meta"]].sample(15, random_state=1)

±1 year matches: 466 | ambiguous, dropped: 4


,title,year,year_meta
425,Ten Canoes,2006,2007
122,Under The Silver Lake,2018,2019
240,Son of a Gun,2014,2015
81,Francofonia,2015,2016
419,Biutiful,2011,2010
590,Amazing Grace,2019,2018
282,Freedom,2014,2015
308,United Passions,2014,2015
83,Alone in Berlin,2016,2017
531,Goldie,2019,2020


## Combining the matches

The exact matches and the one-year matches are combined into one table. We
then check that no `movie_id` appears twice, because `movie_id` is the primary
key of the SALES table, and that no sales row is linked to two movies. Both
checks must show 0.

In [14]:
# Remove the helper columns we no longer need
near = near.drop(columns=["year_meta", "year_difference"])

# Put the two tables under each other
matched = pd.concat([exact, near], ignore_index=True)

print("Total matched:", len(matched))
print("Duplicate movie_id:", matched["movie_id"].duplicated().sum())
print("Duplicate sales url:", matched["url"].duplicated().sum())

Total matched: 8829
Duplicate movie_id: 0
Duplicate sales url: 0


## Cleaning the dates, runtime and money columns

The release date is stored as text like "January 12th", with the year in a
separate column, so we remove the "th", add the year and turn it into a real
date. Values that are not valid numbers or dates (such as "x" or "2020
(Canceled)") become empty. The money columns are stored as whole numbers
(`Int64`), which is needed for the `bigint` columns in PostgreSQL. Missing
values are left empty instead of set to 0, because an unknown budget is not
the same as a budget of zero.

In [15]:
# A function that removes "st", "nd", "rd" or "th" from the end: "12th" -> "12"
def remove_day_ending(day_text):
    for ending in ["st", "nd", "rd", "th"]:
        if day_text.endswith(ending):
            return day_text[:-2]
    return day_text

# "January 12th" -> "January 12" -> "January 12 2001"
date_text = matched["release_date"].astype(str).apply(remove_day_ending)
date_text = date_text + " " + matched["year"].astype(str)

# Turn the text into a real date; invalid dates become empty (errors="coerce")
matched["release_date"] = pd.to_datetime(date_text, format="%B %d %Y", errors="coerce")
print("Unparseable dates:", matched["release_date"].isna().sum())

# Runtime: turn into a number, invalid values become empty
matched["runtime_minutes"] = pd.to_numeric(matched["runtime"], errors="coerce")
matched["runtime_minutes"] = matched["runtime_minutes"].astype("Int64")

# Money columns: turn into whole numbers
money_columns = ["production_budget", "domestic_box_office",
                 "international_box_office", "opening_weekend"]
for column in money_columns:
    matched[column] = pd.to_numeric(matched[column], errors="coerce")
    matched[column] = matched[column].astype("Int64")

print("Missing values:")
print(matched[money_columns].isna().sum())
print("Zero or negative:")
print((matched[money_columns] <= 0).sum())

Unparseable dates: 10
Missing values:
production_budget           5079
domestic_box_office         1363
international_box_office    3142
opening_weekend             1678
dtype: int64
Zero or negative:
production_budget           0
domestic_box_office         0
international_box_office    0
opening_weekend             0
dtype: Int64


## Saving the SALES table

We keep only the columns of the SALES table in our ERD, in the same order, and
sort the rows by `movie_id`. The result has 8,829 movies: the movies for which
both Metacritic data and sales data are available. This table is saved as
`sales.csv` and imported into PostgreSQL.

In [16]:
# Only the columns of the SALES table in the ERD, in the ERD order
sales_table = matched[["movie_id", "release_date", "production_budget", "runtime_minutes",
                       "domestic_box_office", "international_box_office", "opening_weekend"]]

# Sort by movie_id and save
sales_table = sales_table.sort_values("movie_id")
sales_table.to_csv("data_clean/sales.csv", index=False, encoding="utf-8")

print(sales_table.shape)
print("data_clean:", os.listdir("data_clean"))

(8829, 7)
data_clean: ['actors.csv', 'awards.csv', 'critics.csv', 'critic_reviews.csv', 'directors.csv', 'genres.csv', 'movies.csv', 'movies_for_matching.csv', 'movie_actors.csv', 'movie_awards.csv', 'movie_directors.csv', 'movie_genres.csv', 'sales.csv', 'users.csv', 'user_reviews.csv']


## Loading the director, cast, genre and awards data

For the normalization of directors, actors, genres and awards, we need the
columns that we left out of the MOVIES table. We read them again from the raw
Metacritic file and link them to the `movie_id` of each movie using the url,
which is cleaned in exactly the same way as before so that every url matches.
All 11,364 movies are kept, including movies with no director or awards.

In [17]:
# The movie_id and url of every movie in our MOVIES table
movies_ids = pd.read_csv("data_clean/movies.csv", usecols=["movie_id", "url"])

# The four columns we need from the raw Metacritic file
raw_attrs = pd.read_excel("data_raw/metaClean43Brightspace.xlsx", engine="calamine",
                          usecols=["url", "director", "cast", "genre", "awards"])

# Clean the url exactly like in the MOVIES step, so the urls match
raw_attrs["url"] = raw_attrs["url"].str.strip()
raw_attrs["url"] = raw_attrs["url"].str.lower()
raw_attrs["url"] = raw_attrs["url"].str.rstrip("/")

# Link the four columns to the movie_id using the url
# how="left" keeps all movies, also the ones without a director or awards
attrs = movies_ids.merge(raw_attrs, on="url", how="left")

print(attrs.shape, "| movies without a movie_id:", attrs["movie_id"].isna().sum())
attrs.head()

(11364, 6) | movies without a movie_id: 0


,movie_id,url,cast,director,genre,awards
0,1,https://www.metacritic.com/movie/!women-art-re...,NaN,Lynn Hershman-Leeson,Documentary,NaN
1,2,https://www.metacritic.com/movie/10-cloverfiel...,"John Gallagher Jr.,John Goodman,Mary Elizabeth...",Dan Trachtenberg,"Action,Sci-Fi,Drama,Mystery,Thriller,Horror","#18MostDiscussedMovieof2016 , #1MostSharedMovi..."
2,3,https://www.metacritic.com/movie/10-items-or-less,"Jonah Hill,Morgan Freeman,Paz Vega",Brad Silberling,"Drama,Comedy,Romance",NaN
3,4,https://www.metacritic.com/movie/10-years,"Channing Tatum,Chris Pratt,Jenna Dewan",Jamie Linden,"Drama,Comedy,Romance",NaN
4,5,https://www.metacritic.com/movie/100-bloody-acres,NaN,Cameron Cairnes,"Horror,Comedy",NaN


## Normalizing the genres

In the raw data, one cell can contain several genres, such as
"Drama,Comedy,Romance". This breaks the first normal form, which requires one
value per cell. We split the text into separate genres and give every
movie–genre combination its own row. We then create two tables: GENRES, which
stores each genre once with an id, and MOVIE_GENRES, a junction table that
links movies to genres. The junction table is needed because a movie can have
many genres and a genre belongs to many movies (a many-to-many relationship).

In [18]:
# Take the movie_id and genre columns; movies without a genre are removed
pairs = attrs[["movie_id", "genre"]].dropna().copy()

# Split the text into a list: "Drama,Comedy" -> ["Drama", "Comedy"]
pairs["name"] = pairs["genre"].str.split(",")

# Give every genre in the list its own row
pairs = pairs.explode("name")

# Remove spaces around each genre and remove empty values
pairs["name"] = pairs["name"].str.strip()
pairs = pairs[pairs["name"] != ""]

# Keep only movie_id and genre name, and remove double rows
pairs = pairs[["movie_id", "name"]]
pairs = pairs.drop_duplicates()
print("movie-genre pairs:", len(pairs), "| unique genres:", pairs["name"].nunique())

# GENRES table: every genre once, sorted alphabetically, with an id
unique_names = sorted(pairs["name"].unique())
genres = pd.DataFrame()
genres["genre_id"] = range(1, len(unique_names) + 1)
genres["name"] = unique_names

# MOVIE_GENRES table: replace the genre name with its genre_id
movie_genres = pairs.merge(genres, on="name")
movie_genres = movie_genres[["movie_id", "genre_id"]]
movie_genres = movie_genres.sort_values(["movie_id", "genre_id"])

# Save both tables
genres.to_csv("data_clean/genres.csv", index=False, encoding="utf-8")
movie_genres.to_csv("data_clean/movie_genres.csv", index=False, encoding="utf-8")

print(genres.head())
print(movie_genres.head())

movie-genre pairs: 27051 | unique genres: 27
   genre_id       name
0         1     Action
1         2      Adult
2         3  Adventure
3         4  Animation
4         5  Biography
   movie_id  genre_id
0         1         8
1         2         1
3         2         9
6         2        14
4         2        17


## Normalizing directors, actors and awards

Directors, actors (cast) and awards have exactly the same problem as genres:
several values in one cell. Instead of copying the genre code three times, we
put the same steps in a function called `normalize` and use it for each column.
This keeps the code shorter and makes sure all four are cleaned in the same
way. Each call creates a lookup table (for example DIRECTORS) and a junction
table (for example MOVIE_DIRECTORS).

In [19]:
def normalize(column, id_name, table_name, link_name):
    # The same steps as for genres
    pairs = attrs[["movie_id", column]].dropna().copy()
    pairs["name"] = pairs[column].str.split(",")
    pairs = pairs.explode("name")
    pairs["name"] = pairs["name"].str.strip()
    pairs = pairs[pairs["name"] != ""]
    pairs = pairs[["movie_id", "name"]]
    pairs = pairs.drop_duplicates()

    # Lookup table: every name once, with an id
    unique_names = sorted(pairs["name"].unique())
    table = pd.DataFrame()
    table[id_name] = range(1, len(unique_names) + 1)
    table["name"] = unique_names

    # Junction table: movie_id + id
    link = pairs.merge(table, on="name")
    link = link[["movie_id", id_name]]
    link = link.sort_values(["movie_id", id_name])

    # Save both tables
    table.to_csv("data_clean/" + table_name + ".csv", index=False, encoding="utf-8")
    link.to_csv("data_clean/" + link_name + ".csv", index=False, encoding="utf-8")
    print(table_name, len(table), "rows |", link_name, len(link), "rows")

# Use the function for the three columns
normalize("director", "director_id", "directors", "movie_directors")
normalize("cast", "actor_id", "actors", "movie_actors")
normalize("awards", "award_id", "awards", "movie_awards")

directors 6137 rows | movie_directors 11352 rows
actors 24861 rows | movie_actors 48284 rows
awards 6407 rows | movie_awards 6409 rows


## Checking the normalized tables

Before importing the tables into PostgreSQL, we check that no table contains
duplicate rows. The junction tables have a composite primary key (movie_id plus
the second id), so a duplicate row would be rejected by the database. All
counts must be 0.

In [20]:
table_names = ["genres", "movie_genres", "directors", "movie_directors",
               "actors", "movie_actors", "awards", "movie_awards"]

# Open every file and count the duplicate rows
for name in table_names:
    table = pd.read_csv("data_clean/" + name + ".csv")
    print(name, table.shape, "| duplicate rows:", table.duplicated().sum())

genres (27, 2) | duplicate rows: 0
movie_genres (27051, 2) | duplicate rows: 0
directors (6137, 2) | duplicate rows: 0
movie_directors (11352, 2) | duplicate rows: 0
actors (24861, 2) | duplicate rows: 0
movie_actors (48284, 2) | duplicate rows: 0
awards (6407, 2) | duplicate rows: 0
movie_awards (6409, 2) | duplicate rows: 0


## Preparing the review files for import

The critic and user reviews were cleaned by my team members in a separate
notebook. Their file stores `critic_id` and `idvscore` as decimals (70.0),
because pandas converts a whole-number column to decimals as soon as it
contains an empty value, and 24,879 expert reviews have no known critic.
PostgreSQL rejects a decimal for an INT column, so both columns are converted
to whole numbers (`Int64`) before import. The rest of the file is left
unchanged.

In [21]:
# The team's file stores critic_id and idvscore as decimals (70.0), which
# PostgreSQL rejects for INT columns. Convert them to whole numbers.
critic_reviews = pd.read_csv("data_clean/critic_reviews.csv")

critic_reviews["critic_id"] = critic_reviews["critic_id"].astype("Int64")
critic_reviews["idvscore"] = critic_reviews["idvscore"].astype("Int64")

critic_reviews.to_csv("data_clean/critic_reviews.csv", index=False, encoding="utf-8")
print(critic_reviews.shape)

(234389, 10)
